In [36]:
import pandas as pd
import re

# Load the dataset
data = pd.read_csv('../telegram_data.csv')

# Check basic info about the data
print("Initial Dataset Info:")
print(data.info())

# Remove emojis from messages
def remove_emojis(text):
    if pd.isna(text):
        return text
    # General emoji range plus specific problematic emojis
    emoji_pattern = r'[\U0001F000-\U0001FFFF⤵️♨️✅❌]'
    return re.sub(emoji_pattern, '', str(text))

# Apply emoji removal
data['Message'] = data['Message'].apply(remove_emojis)

# Count empty or whitespace-only messages
empty_messages = data['Message'].isna().sum() + data['Message'].str.isspace().sum()
print(f"Number of empty or whitespace-only messages: {empty_messages}")

# Filter rows with Amharic content (ሀ-ፖ Unicode range)
amharic_pattern = r'[\u1200-\u137F]'
data['Amharic_Only'] = data['Message'].apply(lambda x: bool(re.search(amharic_pattern, str(x))) if pd.notnull(x) else False)
amharic_data = data[data['Amharic_Only']]

# Remove unnecessary columns and empty rows
cleaned_data = amharic_data.dropna(subset=['Message']).reset_index(drop=True)

# Save cleaned data
cleaned_data.to_csv('telegram_data_cleaned.csv', index=False)

print(f"Cleaned dataset saved as 'telegram_data_cleaned.csv'. Rows retained: {len(cleaned_data)}")

Initial Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6063 entries, 0 to 6062
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Channel Title     6063 non-null   object
 1   Channel Username  6063 non-null   object
 2   ID                6063 non-null   int64 
 3   Message           5075 non-null   object
 4   Date              6063 non-null   object
 5   Media Path        4027 non-null   object
dtypes: int64(1), object(5)
memory usage: 284.3+ KB
None
Number of empty or whitespace-only messages: 988
Cleaned dataset saved as 'telegram_data_cleaned.csv'. Rows retained: 4406


In [37]:
# Function to clean text
def clean_text(text):
    if isinstance(text, str):
        # Remove emojis (Unicode ranges for emojis)
        text = re.sub(r'[\U0001F000-\U0001FFFF]', '', text)
        
        # Keep Amharic text (Unicode range: 0x1200-0x137F) and Amharic punctuation
        # Remove anything that's not Amharic, whitespace, or Amharic punctuation
        text = re.sub(r'[^\u1200-\u137F\s።፣፤፥፦፧]', '', text)
        
        # Remove extra whitespace
        text = ' '.join(text.split())
        return text
    return ''


In [39]:
import pandas as pd
import re

# Load the cleaned dataset
file_path = '../notebooks/telegram_data_cleaned.csv'
data = pd.read_csv(file_path)

# Function to check if a string contains Amharic text
def contains_amharic(text):
    if pd.isnull(text):
        return False
    amharic_pattern = re.compile(r'[\u1200-\u137F]+')  # Unicode range for Amharic characters
    return bool(amharic_pattern.search(text))

# Filter the dataset for rows where the Message contains Amharic text
data['Is_Amharic'] = data['Message'].apply(contains_amharic)
amharic_data = data[data['Is_Amharic']].drop(columns=['Is_Amharic'])

# Save the filtered dataset
amharic_file_path = 'telegram_data_amharic.csv'
amharic_data.to_csv(amharic_file_path, index=False)

# Output summary
print(f"Total rows in the cleaned dataset: {len(data)}")
print(f"Rows with Amharic messages: {len(amharic_data)}")
print(f"Filtered dataset saved as '{amharic_file_path}'.")

Total rows in the cleaned dataset: 4406
Rows with Amharic messages: 4406
Filtered dataset saved as 'telegram_data_amharic.csv'.


In [41]:
import pandas as pd

# Read the data
df = pd.read_csv('../notebooks/telegram_data_cleaned.csv')

# Print initial shape
print(f"Original shape: {df.shape}")

# Multiple checks for empty messages
df = df[
    (df['Message'].notna()) &  # Remove NaN
    (df['Message'].str.strip() != '') &  # Remove empty strings and whitespace
    (df['Message'].str.len() > 0) &  # Remove zero-length strings
    (~df['Message'].isna())  # Another NaN check
]

# Print final shape
print(f"Shape after cleaning: {df.shape}")

# Display some rows that might still have empty messages to investigate
print("\nChecking potentially empty messages:")
print(df[df['Message'].str.len() < 5])  # Show messages with less than 5 characters

# Save the cleaned data
df.to_csv('telegram_data_cleaned.csv', index=False)

Original shape: (4406, 7)
Shape after cleaning: (4406, 7)

Checking potentially empty messages:
       Channel Title Channel Username    ID Message  \
77            ምርጥ ዕቃ        @MerttEka    56     ሰላም   
2495  Zemen Express®    @ZemenExpress   963    አልቋል   
2665          ምርጥ ዕቃ        @MerttEka  1014    አልቋል   

                           Date Media Path  Amharic_Only  
77    2019-11-16 07:51:19+00:00        NaN          True  
2495  2022-01-21 07:24:15+00:00        NaN          True  
2665  2021-08-18 10:23:15+00:00        NaN          True  


In [42]:
import pandas as pd
import re

# Load the cleaned Amharic dataset
df = pd.read_csv('../notebooks/telegram_data_amharic.csv')

# Tokenize messages
def tokenize_text(text):
    # Use basic whitespace tokenization or an Amharic-specific tokenizer
    return text.split()

# Example function to mock entity labeling
def label_tokens(tokens):
    # For demonstration, randomly assigning "O" (Outside) to all tokens
    # Replace this with actual annotation for your use case
    labels = ['O'] * len(tokens)
    return labels

# Prepare data for CoNLL format
def prepare_conll_data(df):
    conll_data = []
    for _, row in df.iterrows():
        message = row['Message']
        tokens = tokenize_text(message)
        labels = label_tokens(tokens)
        for token, label in zip(tokens, labels):
            conll_data.append(f"{token}\t{label}")
        conll_data.append("")  # Empty line between messages
    return conll_data

# Generate CoNLL data
conll_data = prepare_conll_data(df)

# Save to a text file
with open('amharic_data_conll.txt', 'w', encoding='utf-8') as f:
    f.write("\n".join(conll_data))

print("Data prepared in CoNLL format and saved as 'amharic_data_conll.txt'.")


Data prepared in CoNLL format and saved as 'amharic_data_conll.txt'.


In [43]:
import pandas as pd
import re

# Read the data
df = pd.read_csv('../notebooks/telegram_data_cleaned.csv')

# Print initial shape
print(f"Original shape: {df.shape}")

# Function to clean text
def clean_text(text):
    if isinstance(text, str):
        # Remove emojis and special characters
        text = re.sub(r'[^\w\s.,!?]', '', text)
        # Remove extra whitespace
        text = ' '.join(text.split())
        return text
    return ''

# Clean the messages
df['Message'] = df['Message'].apply(clean_text)

# Remove rows where message is empty after cleaning
df = df[
    (df['Message'].notna()) &
    (df['Message'].str.strip() != '')
]

# Print final shape
print(f"Shape after cleaning: {df.shape}")

# Save the cleaned data
df.to_csv('telegram_data_cleaned.csv', index=False)

# Display sample of cleaned messages
print("\nSample of cleaned messages:")
print(df['Message'].head())

Original shape: (4406, 7)
Shape after cleaning: (4406, 7)

Sample of cleaned messages:
0    ዕቃዎችን ለማዘዝ ኦንላይን helloomarket.com እና ስልክ መደወል ...
1    ZemenExpress is a platform that connect suppli...
2    ZemenExpress is a platform that connect suppli...
3    Super Stretch Silicon Lids 6 pack የዕቃ መሸፈኛ ሲሊከ...
4    Magic Silicone Dish Washing Gloves ዋጋ 400 350ብ...
Name: Message, dtype: object


In [44]:
import re

def label_message_utf8_with_birr(message):
    """
    Function to label tokens in a message for entity extraction.
    - Labels:
        - B-PRODUCT: First token of the product name.
        - I-PRODUCT: Remaining tokens of the product name.
        - I-PRICE: Tokens representing prices.
        - I-LOC: Tokens representing locations.
        - O: Other tokens.
    """
    # Split the message into first line (product name) and the rest
    if '\n' in message:
        first_line, remaining_message = message.split('\n', 1)
    else:
        first_line, remaining_message = message, ""

    labeled_tokens = []

    # Label the first line for product name
    first_line_tokens = re.findall(r'\S+', first_line)
    if first_line_tokens:
        # First token as B-PRODUCT
        labeled_tokens.append(f"{first_line_tokens[0]} B-PRODUCT")
        # Remaining tokens as I-PRODUCT
        for token in first_line_tokens[1:]:
            labeled_tokens.append(f"{token} I-PRODUCT")

    # Process the remaining message for other labels
    if remaining_message:
        lines = remaining_message.split('\n')
        for line in lines:
            tokens = re.findall(r'\S+', line)  # Tokenize each line
            for token in tokens:
                # Check if the token is a price
                if re.match(r'^\d{10,}$', token):
                    labeled_tokens.append(f"{token} O")  # Large numbers as O
                elif re.match(r'^\d+(\.\d{1,2})?$', token) or 'ETB' in token or 'ዋጋ' in token or '$' in token or 'ብር' in token:
                    labeled_tokens.append(f"{token} I-PRICE")
                # Check if the token is a location
                elif any(loc in token for loc in ['Addis Ababa', 'ለቡ', 'መዳህኒዓለም', 'መገናኛ', 'ቦሌ', 'ሜክሲኮ']):
                    labeled_tokens.append(f"{token} I-LOC")
                # Label other tokens as O
                else:
                    labeled_tokens.append(f"{token} O")

    return "\n".join(labeled_tokens)

# Apply the function to the non-null messages
df['Labeled_Message'] = df['Message'].dropna().apply(label_message_utf8_with_birr)

# Display the updated DataFrame with labeled messages
df.head()


,Channel Title,Channel Username,ID,Message,Date,Media Path,Amharic_Only,Labeled_Message
0,HellooMarket,@helloomarketethiopia,10,ዕቃዎችን ለማዘዝ ኦንላይን helloomarket.com እና ስልክ መደወል ...,2019-11-22 11:56:16+00:00,NaN,True,ዕቃዎችን B-PRODUCT\nለማዘዝ I-PRODUCT\nኦንላይን I-PRODU...
1,Zemen Express®,@ZemenExpress,113,ZemenExpress is a platform that connect suppli...,2020-04-23 17:58:01+00:00,NaN,True,ZemenExpress B-PRODUCT\nis I-PRODUCT\na I-PROD...
2,Zemen Express®,@ZemenExpress,115,ZemenExpress is a platform that connect suppli...,2020-04-24 09:32:43+00:00,NaN,True,ZemenExpress B-PRODUCT\nis I-PRODUCT\na I-PROD...
3,AwasMart-አዋስማርት🎁,@AwasMart,2071,Super Stretch Silicon Lids 6 pack የዕቃ መሸፈኛ ሲሊከ...,2022-08-21 11:13:39+00:00,NaN,True,Super B-PRODUCT\nStretch I-PRODUCT\nSilicon I-...
4,AwasMart-አዋስማርት🎁,@AwasMart,2073,Magic Silicone Dish Washing Gloves ዋጋ 400 350ብ...,2022-08-22 04:33:16+00:00,NaN,True,Magic B-PRODUCT\nSilicone I-PRODUCT\nDish I-PR...


In [46]:
import re

def simple_normalize(text):
    # Trim leading and trailing whitespace
    text = text.strip()
    
    # Standardize numerical formats (e.g., ensure numbers have consistent formatting)
    # Replace multiple spaces with a single space
    text = re.sub(r'\s+', ' ', text)
    
    # Ensure numbers are properly formatted (e.g., add spaces around numbers for clarity)
    text = re.sub(r'(\d+)', r' \1 ', text)  # Add spaces around numbers
    text = re.sub(r'\s+', ' ', text)  # Remove extra spaces created by previous step
    
    return text

# Apply the simplified normalization function to the messages
df['Normalized_Message'] = df['Message'].dropna().apply(simple_normalize)

# Save the updated DataFrame
df.to_csv('telegram_data_trimmed_and_numbers_normalized.csv', index=False)

# Display the first few rows of the updated DataFrame
print(df[['Message', 'Normalized_Message']].head())


                                             Message  \
0  ዕቃዎችን ለማዘዝ ኦንላይን helloomarket.com እና ስልክ መደወል ...   
1  ZemenExpress is a platform that connect suppli...   
2  ZemenExpress is a platform that connect suppli...   
3  Super Stretch Silicon Lids 6 pack የዕቃ መሸፈኛ ሲሊከ...   
4  Magic Silicone Dish Washing Gloves ዋጋ 400 350ብ...   

                                  Normalized_Message  
0  ዕቃዎችን ለማዘዝ ኦንላይን helloomarket.com እና ስልክ መደወል ...  
1  ZemenExpress is a platform that connect suppli...  
2  ZemenExpress is a platform that connect suppli...  
3  Super Stretch Silicon Lids 6 pack የዕቃ መሸፈኛ ሲሊከ...  
4  Magic Silicone Dish Washing Gloves ዋጋ 400 350 ...  
